In [11]:
## IMPORTS AND SETUP
# Jupyter Notebook setup
%load_ext autoreload
%autoreload 2

# Imports
import os
import sys
sys.path.insert(0, "/tf/projet") # Add the project root directory to the Python path (docker hosting)

# Imports
import json
import wandb
import datetime
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from Leyanda_Project.utils.warning_clean import silence_tensorflow_warnings
from Leyanda_Project.models.callbacks import create_callbacks
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.applications.inception_v3 import InceptionV3

# Suppress warnings
silence_tensorflow_warnings()

# Check GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU is available: {len(gpus)} device(s) detected")
    except RuntimeError as e:
        print("Error configuring GPU:", str(e))
else:
    print("No GPU available, using CPU")

# Wandb setup
if not os.path.exists("/tf/projet/.env"):
    print("WARNING: No .env file found, please create one with your Wandb API key.")
    exit(1)
else:
    load_dotenv("/tf/projet/.env")
    WANDB_API_KEY = os.getenv("API_KEY")
    wandb_entity = "tom-antoine-cesi"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
TensorFlow warnings suppression is active.
GPU is available: 1 device(s) detected


In [3]:
## PARAMETERS
seed = 123
project_name = "Leyanda"
np.random.seed(seed)
tf.random.set_seed(seed)

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset_livrable_3"  # Path to the raw data folder
images_folder = os.path.join(raw_data_path, "train2017")  # Path to the images folder
annotations_folder = os.path.join(raw_data_path, "annotations")  # Path to the annotations folder

batch_size = 64  # Batch size for dataset loading
max_length = 30  # Maximum caption length
vocab_size_limit = 10000  # Maximum vocabulary size

# Dataset split parameters
train_split = 0.8  # Proportion of the dataset to use for training
val_split = 0.1  # Proportion of the dataset to use for validation
test_split = 0.1  # Proportion of the dataset to use for testing

# Model parameters
embedding_dim = 256  # Dimension of word embeddings
units = 512  # Number of units in LSTM layers

In [22]:
## DATA LOADING AND PREPROCESSING FUNCTIONS
def load_coco_dataset(images_folder, annotations_folder, annotation_file="captions_train2017.json"):
    """
    Load the COCO dataset with images and their captions.

    Parameters:
    ----------
    images_folder : str
        Path to the folder containing images
    annotations_folder : str
        Path to the folder containing annotations
    annotation_file : str, optional
        Name of the annotation file, by default "captions_train2017.json"

    Returns:
    -------
    tuple
        (image_paths, captions) - Lists of image paths and corresponding captions
    """
    print(f"Loading COCO dataset from {images_folder} and {annotations_folder}...")

    annotations_path = os.path.join(annotations_folder, annotation_file)
    with open(annotations_path, 'r') as f:
        annotations_data = json.load(f)

    image_paths = []
    captions = []

    for annotation in annotations_data['annotations']:
        img_id = annotation['image_id']
        img_name = f'{int(img_id):012d}.jpg'
        img_path = os.path.join(images_folder, img_name)

        if os.path.exists(img_path):
            image_paths.append(img_path)
            captions.append(annotation['caption'])

    print(f"Loaded {len(image_paths)} images with captions")
    return image_paths, captions

def create_tokenizer(captions, num_words=10000):
    """
    Create and fit a tokenizer on all captions.

    Parameters:
    ----------
    captions : list
        List of all captions
    num_words : int, optional
        Maximum number of words to keep, by default 10000

    Returns:
    -------
    tuple
        (tokenizer, vocab_size) - Fitted tokenizer and vocabulary size
    """
    print("Creating and fitting tokenizer...")

    tokenizer = Tokenizer(
        num_words=num_words,
        oov_token="<unk>",
        filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ '
    )

    processed_captions = ['<start> ' + caption + ' <end>' for caption in captions]

    tokenizer.fit_on_texts(processed_captions)

    word_index = tokenizer.word_index
    if '<start>' not in word_index:
        word_index['<start>'] = len(word_index) + 1
    if '<end>' not in word_index:
        word_index['<end>'] = len(word_index) + 1

    vocab_size = min(num_words, len(tokenizer.word_index) + 1)
    print(f"Vocabulary size: {vocab_size}")

    return tokenizer, vocab_size


def preprocess_image_path(img_path, target_size=(299, 299)):
    """
    Load and preprocess an image from path.

    Parameters:
    ----------
    img_path : str
        Path to the image
    target_size : tuple, optional
        Target size for resizing, by default (299, 299)

    Returns:
    -------
    tensor
        Preprocessed image tensor
    """
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, target_size)
    img = preprocess_input(img)
    return img


def preprocess_caption(caption, tokenizer, max_length=30):
    """
    Preprocess a caption: add tokens, convert to sequence and pad.

    Parameters:
    ----------
    caption : str
        Caption text
    tokenizer : Tokenizer
        Fitted tokenizer
    max_length : int, optional
        Maximum caption length, by default 30

    Returns:
    -------
    ndarray
        Tokenized and padded caption
    """
    caption = '<start> ' + caption + ' <end>'

    sequence = tokenizer.texts_to_sequences([caption])[0]
    padded_sequence = pad_sequences([sequence], maxlen=max_length, padding='post')[0]

    return padded_sequence


def create_dataset_generator(image_paths, captions, tokenizer, max_length=30, batch_size=32, shuffle=True):
    """
    Create a TensorFlow data generator that yields batches of preprocessed images and captions.
    Parameters:
    ----------
    image_paths : list
        List of image paths
    captions : list
        List of corresponding captions
    tokenizer : Tokenizer
        Fitted tokenizer
    max_length : int, optional
        Maximum caption length, by default 30
    batch_size : int, optional
        Batch size, by default 32
    shuffle : bool, optional
        Whether to shuffle the dataset, by default True
    Returns:
    -------
    tf.data.Dataset
        TensorFlow dataset that yields ([image, caption_input], caption_target) pairs
    """
    def generator():
        indices = list(range(len(image_paths)))
        if shuffle:
            np.random.shuffle(indices)

        for i in indices:
            img_path = image_paths[i]
            caption = captions[i]

            img = preprocess_image_path(img_path, target_size=(299, 299))
            seq = preprocess_caption(caption, tokenizer, max_length)

            yield [img, seq], seq

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            (
                tf.TensorSpec(shape=(299, 299, 3), dtype=tf.float32),
                tf.TensorSpec(shape=(max_length,), dtype=tf.int32)
            ),
            tf.TensorSpec(shape=(max_length,), dtype=tf.int32)
        )
    )

    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)

    return dataset

def split_dataset(image_paths, captions, train_split=0.8, val_split=0.1):
    """
    Split the dataset into training, validation and test sets.

    Parameters:
    ----------
    image_paths : list
        List of image paths
    captions : list
        List of corresponding captions
    train_split : float, optional
        Proportion for training, by default 0.8
    val_split : float, optional
        Proportion for validation, by default 0.1

    Returns:
    -------
    tuple
        (train_img_paths, train_captions, val_img_paths, val_captions, test_img_paths, test_captions)
    """
    print("Splitting dataset into train, validation, and test sets...")

    indices = np.arange(len(image_paths))
    np.random.shuffle(indices)

    train_size = int(train_split * len(image_paths))
    val_size = int(val_split * len(image_paths))

    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size+val_size]
    test_indices = indices[train_size+val_size:]

    train_img_paths = [image_paths[i] for i in train_indices]
    train_captions = [captions[i] for i in train_indices]

    val_img_paths = [image_paths[i] for i in val_indices]
    val_captions = [captions[i] for i in val_indices]

    test_img_paths = [image_paths[i] for i in test_indices]
    test_captions = [captions[i] for i in test_indices]

    print(f"Train set: {len(train_img_paths)} samples")
    print(f"Validation set: {len(val_img_paths)} samples")
    print(f"Test set: {len(test_img_paths)} samples")

    return (train_img_paths, train_captions,
            val_img_paths, val_captions,
            test_img_paths, test_captions)

In [23]:
## DATA PREPARATION WORKFLOW
print(f"Starting data preparation workflow for {project_name}...")

# Step 1: Load COCO dataset
image_paths, captions = load_coco_dataset(
    images_folder=images_folder,
    annotations_folder=annotations_folder
)

# Step 2: Limit dataset size for testing (remove for full training)
max_samples = 10000
if len(image_paths) > max_samples:
    print(f"Limiting dataset to {max_samples} samples for testing")
    random_indices = np.random.choice(len(image_paths), max_samples, replace=False)
    image_paths = [image_paths[i] for i in random_indices]
    captions = [captions[i] for i in random_indices]

# Step 3: Create and fit tokenizer
tokenizer, vocab_size = create_tokenizer(captions, num_words=vocab_size_limit)

# Step 4: Split dataset
(train_img_paths, train_captions,
 val_img_paths, val_captions,
 test_img_paths, test_captions) = split_dataset(
    image_paths,
    captions,
    train_split=train_split,
    val_split=val_split
)

# Step 5: Create TensorFlow datasets
print("Creating TensorFlow datasets...")
train_dataset = create_dataset_generator(
    train_img_paths,
    train_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size
)

val_dataset = create_dataset_generator(
    val_img_paths,
    val_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size
)

test_dataset = create_dataset_generator(
    test_img_paths,
    test_captions,
    tokenizer,
    max_length=max_length,
    batch_size=batch_size
)

print("Data preparation complete!")

for images, captions in train_dataset.take(1):
    print(f"Image batch shape: {images.shape}")
    print(f"Caption batch shape: {captions.shape}")
    break

Starting data preparation workflow for Leyanda...
Loading COCO dataset from /tf/projet/Dataset_livrable_3/train2017 and /tf/projet/Dataset_livrable_3/annotations...
Loaded 591753 images with captions
Limiting dataset to 10000 samples for testing
Creating and fitting tokenizer...
Vocabulary size: 5042
Splitting dataset into train, validation, and test sets...
Train set: 8000 samples
Validation set: 1000 samples
Test set: 1000 samples
Creating TensorFlow datasets...
Data preparation complete!


InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_3_device_/job:localhost/replica:0/task:0/device:CPU:0}} TypeError: `generator` yielded an element that did not match the expected structure. The expected structure was ((tf.float32, tf.int32), tf.int32), but the yielded element was ([<tf.Tensor: shape=(299, 299, 3), dtype=float32, numpy=
array([[[ 1.        ,  1.        ,  1.        ],
        [ 1.        ,  1.        ,  1.        ],
        [ 1.        ,  1.        ,  1.        ],
        ...,
        [ 0.6582726 ,  0.6269001 ,  0.5406256 ],
        [ 0.66243505,  0.6310625 ,  0.544788  ],
        [ 0.65937436,  0.6280018 ,  0.5417273 ]],

       [[ 1.        ,  1.        ,  1.        ],
        [ 1.        ,  1.        ,  1.        ],
        [ 1.        ,  1.        ,  1.        ],
        ...,
        [ 0.654902  ,  0.62352943,  0.5372549 ],
        [ 0.6606847 ,  0.62931216,  0.54303765],
        [ 0.6643274 ,  0.63295484,  0.54668033]],

       [[ 1.   

In [19]:
## MODELS FUNCTIONS

def create_image_encoder(input_shape=(299, 299, 3)):
    """
    Create an image encoder based on InceptionV3 pre-trained model.
    Parameters:
    ----------
    input_shape : tuple, optional
        Shape of input images, by default (299, 299, 3)
    Returns:
    -------
    Model
        Image encoder model that extracts features from images
    """
    base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=input_shape)

    for layer in base_model.layers:
        layer.trainable = False

    output = base_model.output
    output = tf.keras.layers.GlobalAveragePooling2D()(output)
    output = tf.keras.layers.Dense(embedding_dim, activation='relu')(output)
    encoder = Model(inputs=base_model.input, outputs=output)

    return encoder


def create_caption_decoder(vocab_size, max_length, embedding_dim, units):
    """
    Create a decoder model that generates captions from image features.
    Parameters:
    ----------
    vocab_size : int
        Size of the vocabulary
    max_length : int
        Maximum length of captions
    embedding_dim : int
        Dimension of word embeddings
    units : int
        Number of units in LSTM layers
    Returns:
    -------
    Model
        Caption decoder model
    """
    image_features = Input(shape=(embedding_dim,))
    caption_input = Input(shape=(max_length,))
    embedding = Embedding(input_dim=vocab_size,
                          output_dim=embedding_dim,
                          mask_zero=True)(caption_input)

    image_features_dense = Dense(units, activation='relu')(image_features)
    lstm = LSTM(units, return_sequences=True)(embedding, initial_state=[image_features_dense, image_features_dense])

    dropout = Dropout(0.3)(lstm)
    output = Dense(vocab_size, activation='softmax')(dropout)
    decoder = Model(inputs=[image_features, caption_input], outputs=output)

    return decoder


def create_captioning_model(encoder, decoder, max_length):
    """
    Create the complete image captioning model by connecting encoder and decoder.

    Parameters:
    ----------
    encoder : Model
        Image encoder model
    decoder : Model
        Caption decoder model
    max_length : int
        Maximum length of captions

    Returns:
    -------
    Model
        Complete image captioning model
    """
    input_tensor = Input(shape=(299, 299, 3))

    image_features = encoder(input_tensor)

    caption_input = Input(shape=(max_length,))

    caption_output = decoder([image_features, caption_input])

    captioning_model = Model(inputs=[input_tensor, caption_input],
                            outputs=caption_output)

    return captioning_model

def create_inference_model(encoder, decoder, max_length, vocab_size):
    """
    Create a model for inference (generating captions for new images).
    Parameters:
    ----------
    encoder : Model
        Image encoder model
    decoder : Model
        Caption decoder model
    max_length : int
        Maximum length of captions
    vocab_size : int
        Size of the vocabulary
    Returns:
    -------
    tuple
        (encoder_model, decoder_model) - Models for inference
    """
    encoder_model = encoder
    image_features_input = Input(shape=(embedding_dim,))
    decoder_state_input_h = Input(shape=(units,))
    decoder_state_input_c = Input(shape=(units,))
    decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
    decoder_input = Input(shape=(1,))
    embedding_layer = decoder.layers[2]
    embedding = embedding_layer(decoder_input)
    lstm_layer = decoder.layers[4]
    lstm_outputs, state_h, state_c = lstm_layer(embedding, initial_state=decoder_states_inputs)
    decoder_states = [state_h, state_c]
    dense_layer = decoder.layers[-1]
    decoder_outputs = dense_layer(lstm_outputs)

    decoder_model = Model(
        inputs=[decoder_input, image_features_input, decoder_state_input_h, decoder_state_input_c],
        outputs=[decoder_outputs] + decoder_states
    )

    return encoder_model, decoder_model


def generate_caption(image_path, encoder_model, decoder_model, tokenizer, max_length):
    """
    Generate a caption for a given image.
    Parameters:
    ----------
    image_path : str
        Path to the image
    encoder_model : Model
        Encoder model for feature extraction
    decoder_model : Model
        Decoder model for caption generation
    tokenizer : Tokenizer
        Tokenizer used to convert words to indices and vice versa
    max_length : int
        Maximum length of generated caption
    Returns:
    -------
    str
        Generated caption
    """
    img = preprocess_image_path(image_path)
    img = np.expand_dims(img, axis=0)
    image_features = encoder_model.predict(img)
    decoder_input = np.zeros((1, 1))
    decoder_input[0, 0] = tokenizer.word_index['<start>']
    decoder_h = np.zeros((1, units))
    decoder_c = np.zeros((1, units))
    generated_caption = []

    for i in range(max_length):
        predictions, decoder_h, decoder_c = decoder_model.predict(
            [decoder_input, image_features, decoder_h, decoder_c]
        )
        predicted_id = np.argmax(predictions[0, 0])
        predicted_word = None
        for word, index in tokenizer.word_index.items():
            if index == predicted_id:
                predicted_word = word
                break

        if predicted_word == '<end>' or predicted_word is None:
            break

        if predicted_word not in ['<start>', '<pad>']:
            generated_caption.append(predicted_word)

        decoder_input[0, 0] = predicted_id

    return ' '.join(generated_caption)


def loss_function(real, pred):
    """
    Custom loss function for caption generation that masks padding tokens.
    Parameters:
    ----------
    real : Tensor
        Ground truth captions
    pred : Tensor
        Predicted captions
    Returns:
    -------
    Tensor
        Masked loss value
    """
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)


def plot_training_history(history):
    """
    Plot the training and validation loss and accuracy.
    Parameters:
    ----------
    history : History
        Training history
    """
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss Value')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()


def show_example_captions(test_img_paths, test_captions, encoder, decoder, tokenizer, max_length, num_examples=5):
    """
    Display example images with their actual and predicted captions.
    Parameters:
    ----------
    test_img_paths : list
        List of image paths
    test_captions : list
        List of actual captions
    encoder : Model
        Encoder model
    decoder : Model
        Decoder model
    tokenizer : Tokenizer
        Tokenizer for word conversion
    max_length : int
        Maximum caption length
    num_examples : int, optional
        Number of examples to show, by default 5
    """
    encoder_model, decoder_model = create_inference_model(encoder, decoder, max_length, len(tokenizer.word_index) + 1)

    indices = np.random.choice(len(test_img_paths), num_examples, replace=False)
    plt.figure(figsize=(15, 25))

    for i, idx in enumerate(indices):
        img_path = test_img_paths[idx]
        actual_caption = test_captions[idx]
        predicted_caption = generate_caption(img_path, encoder_model, decoder_model, tokenizer, max_length)
        plt.subplot(num_examples, 1, i+1)
        img = plt.imread(img_path)
        plt.imshow(img)
        plt.title(f'Actual: {actual_caption}\nPredicted: {predicted_caption}', fontsize=12)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [20]:
## MODELS CREATION WORKFLOW

print(f"Starting model creation workflow for {project_name}...")

# Initialize weights & biases
wandb.init(
    project=project_name,
    name=f"captioning_model_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}",
    entity=wandb_entity,
    config={
        "architecture": "InceptionV3-LSTM",
        "dataset": "COCO",
        "embedding_dim": embedding_dim,
        "units": units,
        "max_length": max_length,
        "vocab_size": len(tokenizer.word_index) + 1,
        "batch_size": batch_size
    }
)

# Create encoder model
print("Creating image encoder...")
encoder = create_image_encoder()

# Create decoder model
vocab_size = len(tokenizer.word_index) + 1
print(f"Creating caption decoder with vocabulary size: {vocab_size}")
decoder = create_caption_decoder(vocab_size, max_length, embedding_dim, units)

# Create the complete captioning model
print("Creating captioning model...")
captioning_model = create_captioning_model(encoder, decoder, max_length)

# Compile the model
print("Compiling the model...")
captioning_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=loss_function,
    metrics=['accuracy']
)

# Display model summary
print("Model architecture summary:")
captioning_model.summary()

print("Models created successfully!")

Starting model creation workflow for Leyanda...


Creating image encoder...
Creating caption decoder with vocabulary size: 5069
Creating captioning model...
Compiling the model...
Model architecture summary:


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_13      │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_6        │ (None, 256)       │ 22,327,328 │ input_layer_13[0… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_14      │ (None, 30)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_7        │ (None, 30, 5069)  │  5,604,557 │ functional_6[0][… │
│ (Functional)        │                   │            │ input_layer_14[0… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 27,931,885 (106.55 MB)

 Trainable params: 6,129,101 (23.38 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

Models created successfully!


In [21]:
## MODELS TRAINING WORKFLOW

print(f"Starting model training workflow for {project_name}...")

# Create callbacks with wandb integration
callbacks = create_callbacks(
    model_name=f"{project_name}_captioning",
    tensorboard=True,
    early_stopping=True,
    model_checkpoint=True
)

# Add WandB callback
wandb_callback = wandb.keras.WandbMetricsLogger()
callbacks.append(wandb_callback)

# Set number of epochs
epochs = 15  # Ajuste selon tes contraintes de temps

# Train the model
print(f"Training the model for {epochs} epochs...")
history = captioning_model.fit(
    train_dataset,
    epochs=epochs,
    validation_data=val_dataset,
    callbacks=callbacks
)

# Plot training history
plot_training_history(history)

# Evaluate the model on test dataset
print("Evaluating the model on test dataset...")
results = captioning_model.evaluate(test_dataset)
print(f"Test Loss: {results[0]:.4f}")
print(f"Test Accuracy: {results[1]:.4f}")

# Log final metrics to wandb
wandb.log({
    "final_test_loss": results[0],
    "final_test_accuracy": results[1]
})

# Save the models
print("Saving models...")
captioning_model.save(f"{project_name}_captioning_model.h5")
encoder.save(f"{project_name}_encoder.h5")
decoder.save(f"{project_name}_decoder.h5")

print("Model training complete!")

Starting model training workflow for Leyanda...
Creating callbacks...
Training the model for 15 epochs...
Epoch 1/15


ValueError: Layer "functional_8" expects 2 input(s), but it received 1 input tensors. Inputs received: [<tf.Tensor 'data:0' shape=(None, 299, 299, 3) dtype=float32>]